[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlog7997/rag-playbook/blob/master/learn/06-how-do-i-know-its-working/notebook.ipynb)

# Chapter 6 — How Do I Know It's Actually Working?

## The Real Story

After building 6 phases of RAG improvements — hybrid search, reranking, contextual embeddings, multi-query, guardrails — we ran formal evaluation on the healthcare assistant. The results were humbling.

**The finding that changed everything:** Hybrid search beat Naive by **20% on Recall@1**. But Contextual Embeddings only added marginal improvement over Hybrid. We had spent significant time on contextual embeddings optimization and the payoff was minimal.

The bottleneck wasn't reranking or embedding quality. It was **retrieval** — specifically, the semantic gap problem we addressed in Chapter 2. Fixing that first with hybrid search was the right call. Everything else was incremental.

**This is why formal evaluation matters. Without numbers, you're optimizing by intuition. With numbers, you know exactly where the leverage is.**

---

## Results from the Real System (Adapted)

```
Phase               Recall@1  Recall@3  Recall@5   MRR
─────────────────────────────────────────────────────
Naive (Vector)        0.30      0.60      0.80    0.508
Hybrid (RRF)          0.50      0.70      0.90    0.628
Contextual            0.50      0.70      0.90    0.650
```

Hybrid added +20% Recall@1. Contextual added 0%. The bottleneck insight: **good reranking on bad retrieval still gives bad answers. Fix retrieval first.**

## What You'll Learn
- Build an evaluation dataset with Claude Sonnet
- Recall@1, Recall@3, Recall@5, and MRR — what they mean and how to compute them
- LLM judge: faithfulness, relevance, completeness scores
- RAGAS overview and how to interpret the metrics

## Setup

In [ ]:
# Install required packages
# For RAGAS (optional): pip install ragas langchain-anthropic
!pip install anthropic==0.49.0 chromadb sentence-transformers rank-bm25 keyring -q

In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

import keyring
# keyring.set_password("Anthorpic", "API_KEY", "your-key-here")  # run once
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
LEARN_DIR = REPO_ROOT / 'learn'
sys.path.insert(0, str(LEARN_DIR))

print(f"Repo root: {REPO_ROOT}")
print("Setup complete!")

## Step 1: Build an Evaluation Dataset

The evaluation dataset is everything. **Bad eval = misleading results.**

**Rules for building good eval questions:**
1. Use **different words** than the docs — test semantic search, not keyword matching
2. Include all semantic gap cases — "cancel" when docs say "terminate"
3. Include easy questions too — to ensure you haven't broken basic cases
4. Specify the exact source section — so you can verify retrieval precisely
5. Generate with LLM, validate manually

We ask Claude Sonnet to read each doc section and generate questions that a **real user might ask** — using different phrasing than the docs.

In [ ]:
import json
import re
from pathlib import Path
from utils import ask_claude

KB_DIR = REPO_ROOT / "knowledge-base"
DATA_DIR = REPO_ROOT / "data"
EVAL_PATH = DATA_DIR / "eval_dataset.json"

DATA_DIR.mkdir(exist_ok=True)

QUESTION_GEN_SYSTEM = """You are creating an evaluation dataset for testing a RAG system.
Given a documentation section, generate 10 realistic user questions with expected answers.

CRITICAL RULES:
1. Questions MUST use DIFFERENT words than the documentation (test semantic search)
   - If docs say "subscription termination" -> ask about "cancelling my account"
   - If docs say "member offboarding" -> ask about "removing someone from the team"
   - If docs say "account recovery via email verification token flow" -> ask "forgot password"
2. Questions should be what a REAL USER would type into a support chat
3. Expected answers should be concise (1-3 sentences) based on the doc content
4. Mix difficulty: 3 easy (direct), 4 medium (paraphrase), 3 hard (semantic gap)

Return ONLY a JSON array:
[
  {
    "question": "...",
    "expected_answer": "...",
    "source_file": "filename.md",
    "source_section": "Section Name",
    "difficulty": "easy|medium|hard"
  },
  ...
]"""


def generate_qa_pairs(filename: str, content: str) -> list:
    """Generate Q&A pairs for a single knowledge base file."""
    response = ask_claude(
        system=QUESTION_GEN_SYSTEM,
        user=f"""Documentation file: {filename}

Content:
{content[:3000]}

Generate 10 evaluation questions with expected answers. Remember: use DIFFERENT
words than the documentation to test semantic search.""",
        model="claude-sonnet-4-6",  # Sonnet for better question quality
    )

    try:
        clean = re.sub(r"```json\s*|\s*```", "", response).strip()
        pairs = json.loads(clean)
        if isinstance(pairs, list):
            for pair in pairs:
                pair["source_file"] = filename
            return pairs
    except (json.JSONDecodeError, ValueError):
        print(f"  Parse error for {filename}")
        return []


print("Eval dataset functions defined.")
print(f"Knowledge base directory: {KB_DIR}")
print(f"Will save to: {EVAL_PATH}")

In [ ]:
# Check if dataset already exists — if yes, load it; if no, build it
# Building costs ~$0.05 (10 Sonnet calls). Run once and reuse.

if EVAL_PATH.exists():
    existing = json.loads(EVAL_PATH.read_text(encoding='utf-8'))
    if len(existing) > 0:
        print(f"Existing dataset found with {len(existing)} Q&A pairs.")
        print("Loading existing dataset...")
        all_qa_pairs = existing
    else:
        all_qa_pairs = None
else:
    all_qa_pairs = None

if all_qa_pairs is None:
    print("No dataset found. Building now...")
    print("(This makes ~10 Claude Sonnet API calls — may take 1-2 minutes)")
    all_qa_pairs = []
    md_files = sorted(KB_DIR.glob("*.md"))
    print(f"Found {len(md_files)} knowledge base files.\n")

    for i, md_file in enumerate(md_files, 1):
        print(f"[{i}/{len(md_files)}] Generating questions for {md_file.name}...")
        content = md_file.read_text(encoding="utf-8")
        pairs = generate_qa_pairs(md_file.name, content)
        print(f"  Generated {len(pairs)} Q&A pairs")
        if pairs:
            print(f"  Sample: '{pairs[0]['question']}' [{pairs[0].get('difficulty', 'unknown')}]")
        all_qa_pairs.extend(pairs)

    # Save to disk
    with open(EVAL_PATH, "w", encoding="utf-8") as f:
        json.dump(all_qa_pairs, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {len(all_qa_pairs)} Q&A pairs to {EVAL_PATH}")

print(f"\nDataset ready: {len(all_qa_pairs)} questions")

In [ ]:
# Dataset statistics
from collections import Counter

by_source = Counter(p["source_file"] for p in all_qa_pairs)
by_difficulty = Counter(p.get("difficulty", "unknown") for p in all_qa_pairs)

print("Dataset Statistics")
print("=" * 65)

print("\nBy source file:")
for source, count in sorted(by_source.items()):
    print(f"  {source}: {count} questions")

print("\nBy difficulty:")
for diff, count in sorted(by_difficulty.items()):
    print(f"  {diff}: {count} questions")

print(f"\nTotal: {len(all_qa_pairs)} questions across {len(by_source)} files")

# Show some hard questions (semantic gap test cases)
hard_questions = [p for p in all_qa_pairs if p.get("difficulty") == "hard"]
if hard_questions:
    print(f"\nExample hard questions (semantic gap test cases):")
    for q in hard_questions[:5]:
        print(f"  [{q['source_file']}] {q['question']}")

## Step 2: Recall@K and MRR

### What these metrics mean:

**Recall@K:** "In the top K retrieved chunks, did the correct answer appear?"
- Recall@1: Correct chunk is ranked #1 — most important for user experience
- Recall@3: Correct chunk is in top 3 — reasonable for LLM synthesis
- Recall@5: Correct chunk is in top 5 — upper bound of typical fetch size

**MRR (Mean Reciprocal Rank):** "On average, where does the correct chunk appear?"
```
MRR = (1/N) * Σ  1 / rank(correct chunk)
```
- Always at rank 1 → MRR = 1.0
- Always at rank 2 → MRR = 0.5  
- Always at rank 3 → MRR = 0.33

MRR rewards systems that consistently put the correct answer near the top.

In [ ]:
from utils import vector_search, hybrid_search


def get_rank(hits: list, expected_source: str, expected_section: str) -> int:
    """
    Find the rank (1-based) of the correct chunk in the retrieved hits.
    Uses flexible section matching (substring) to handle section name variations.
    Returns -1 if not found in the top K results.
    """
    expected_section_lower = expected_section.lower()
    best_rank = -1
    for rank, hit in enumerate(hits, 1):
        if hit["source"] != expected_source:
            continue
        # Section match: exact, substring, or reverse substring
        if (expected_section_lower == hit["section"].lower()
                or expected_section_lower in hit["section"].lower()
                or hit["section"].lower() in expected_section_lower):
            return rank
        # Track best source-only match as fallback
        if best_rank == -1:
            best_rank = rank
    return best_rank  # Fall back to source-only match


def compute_metrics(eval_pairs: list, search_fn, search_name: str, top_k: int = 5) -> dict:
    """Run a search function over all eval pairs and compute retrieval metrics."""
    results = []
    reciprocal_ranks = []

    print(f"  Running {search_name} on {len(eval_pairs)} questions...", end="", flush=True)

    for pair in eval_pairs:
        hits = search_fn(pair["question"], top_k=top_k)
        rank = get_rank(hits, pair["source_file"], pair.get("source_section", ""))

        rr = 1.0 / rank if rank > 0 else 0.0
        reciprocal_ranks.append(rr)
        results.append({
            "question": pair["question"],
            "source_file": pair["source_file"],
            "source_section": pair.get("source_section", ""),
            "difficulty": pair.get("difficulty", "unknown"),
            "rank": rank,
            "correct_at_1": rank == 1,
            "correct_at_3": 0 < rank <= 3,
            "correct_at_5": 0 < rank <= 5,
            "rr": rr,
        })

    print(" Done.")

    n = len(results)
    return {
        "name": search_name,
        "recall_at_1": sum(r["correct_at_1"] for r in results) / n,
        "recall_at_3": sum(r["correct_at_3"] for r in results) / n,
        "recall_at_5": sum(r["correct_at_5"] for r in results) / n,
        "mrr": sum(reciprocal_ranks) / n,
        "per_question": results,
    }


print("=" * 65)
print("  Step 2: Retrieval Evaluation — Recall@K and MRR")
print("=" * 65)
print(f"\nEvaluating on {len(all_qa_pairs)} questions...\n")

naive_results = compute_metrics(
    all_qa_pairs,
    search_fn=lambda q, top_k: vector_search(q, top_k=top_k),
    search_name="Naive Vector",
    top_k=5,
)

hybrid_results = compute_metrics(
    all_qa_pairs,
    search_fn=lambda q, top_k: hybrid_search(q, top_k=top_k, fetch_k=top_k * 2),
    search_name="Hybrid RRF",
    top_k=5,
)

In [ ]:
# Print comparison table
print("\n" + "=" * 65)
print("  Retrieval Metrics Comparison")
print("=" * 65)
print(f"\n{'Phase':<22} {'Recall@1':>9} {'Recall@3':>9} {'Recall@5':>9} {'MRR':>9}")
print(f"{'-' * 22} {'-' * 9} {'-' * 9} {'-' * 9} {'-' * 9}")

for res in [naive_results, hybrid_results]:
    print(
        f"{res['name']:<22} "
        f"{res['recall_at_1']:>9.3f} "
        f"{res['recall_at_3']:>9.3f} "
        f"{res['recall_at_5']:>9.3f} "
        f"{res['mrr']:>9.3f}"
    )

# Delta row
d1 = hybrid_results["recall_at_1"] - naive_results["recall_at_1"]
d3 = hybrid_results["recall_at_3"] - naive_results["recall_at_3"]
d5 = hybrid_results["recall_at_5"] - naive_results["recall_at_5"]
dm = hybrid_results["mrr"] - naive_results["mrr"]
print(f"{'-' * 22} {'-' * 9} {'-' * 9} {'-' * 9} {'-' * 9}")
print(f"{'Hybrid improvement':<22} {d1:>+9.3f} {d3:>+9.3f} {d5:>+9.3f} {dm:>+9.3f}")

print()
if d1 > 0:
    print(f"Hybrid improved Recall@1 by {d1:+.1%} — confirms Chapter 2 findings.")
print("\nKey insight: These numbers are RETRIEVAL only — correct chunk was retrieved.")
print("A system can have good retrieval and still generate poor answers.")
print("Use LLM judge (next) to evaluate end-to-end quality.")

In [ ]:
# Breakdown by difficulty — shows WHERE each approach fails
print("\nRecall@1 Breakdown by Question Difficulty")
print("=" * 65)

for diff in ["easy", "medium", "hard"]:
    naive_diff = [r for r in naive_results["per_question"] if r.get("difficulty") == diff]
    hybrid_diff = [r for r in hybrid_results["per_question"] if r.get("difficulty") == diff]

    if naive_diff:
        naive_r1 = sum(r["correct_at_1"] for r in naive_diff) / len(naive_diff)
        hybrid_r1 = sum(r["correct_at_1"] for r in hybrid_diff) / len(hybrid_diff)
        print(f"\n  {diff.capitalize()} ({len(naive_diff)} questions):")
        print(f"    Naive:  {naive_r1:.3f}")
        print(f"    Hybrid: {hybrid_r1:.3f}  (delta: {hybrid_r1 - naive_r1:+.3f})")

print()
print("Expected: biggest delta on 'hard' questions (semantic gap cases).")
print("Easy questions should be similar — both approaches handle direct language well.")

## Step 3: LLM Judge — Answer Quality Evaluation

Retrieval metrics only measure whether the right chunk was found. To measure **answer quality**, we use an LLM judge.

For each answer, we score on three dimensions:
- **Faithfulness (1-5):** Does the answer only use information from the retrieved context? (hallucination detection)
- **Relevance (1-5):** Does the answer directly address the question?
- **Completeness (1-5):** Does the answer cover all aspects of the question?

Verdict: `pass` if all three scores >= 3, `fail` otherwise.

In [ ]:
from utils import hybrid_search, ask_claude

ANSWER_SYSTEM = """You are a helpful NovaDash support assistant.
Answer the question using ONLY the documentation provided.
Be specific and cite exact details from the docs where possible.
If the answer isn't in the docs, say "I don't have that information." """

JUDGE_SYSTEM = """You are an expert evaluator for an AI question-answering system.
You will receive:
  1. A user question
  2. The expected answer (from ground truth)
  3. The retrieved documentation context
  4. The system-generated answer

Evaluate the generated answer on three dimensions:
  - faithfulness (1-5): Does the answer only use information from the context?
    5 = completely grounded, 1 = makes up information not in context
  - relevance (1-5): Does the answer address the question?
    5 = directly and fully answers, 1 = doesn't address the question
  - completeness (1-5): Does the answer cover all key aspects?
    5 = comprehensive, 1 = misses major aspects

Return ONLY a JSON object:
{
  "faithfulness": <1-5>,
  "relevance": <1-5>,
  "completeness": <1-5>,
  "verdict": "<pass|fail>",
  "notes": "<one sentence explaining the verdict>"
}

Verdict rules:
  pass: all three scores >= 3
  fail: any score < 3"""


def generate_answer(question: str):
    hits = hybrid_search(question, top_k=4)
    context = "\n\n---\n\n".join([
        f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
        for h in hits
    ])
    answer = ask_claude(
        system=ANSWER_SYSTEM,
        user=f"Documentation context:\n{context}\n\nQuestion: {question}",
        model="claude-haiku-4-5-20251001",
    )
    return answer, context


def judge_answer(question: str, expected: str, context: str, answer: str) -> dict:
    """Ask Claude Haiku to judge the quality of an answer."""
    import re
    import json
    response = ask_claude(
        system=JUDGE_SYSTEM,
        user=f"""Question: {question}

Expected answer: {expected}

Retrieved context (first 800 chars):
{context[:800]}

Generated answer:
{answer}

Evaluate the generated answer.""",
        model="claude-haiku-4-5-20251001",
    )

    try:
        clean = re.sub(r"```json\s*|\s*```", "", response).strip()
        result = json.loads(clean)
        return {
            "faithfulness": int(result.get("faithfulness", 3)),
            "relevance": int(result.get("relevance", 3)),
            "completeness": int(result.get("completeness", 3)),
            "verdict": result.get("verdict", "fail"),
            "notes": result.get("notes", ""),
        }
    except (json.JSONDecodeError, ValueError):
        f_match = re.search(r'"faithfulness"\s*:\s*(\d)', response)
        r_match = re.search(r'"relevance"\s*:\s*(\d)', response)
        c_match = re.search(r'"completeness"\s*:\s*(\d)', response)
        v_match = re.search(r'"verdict"\s*:\s*"(\w+)"', response)
        return {
            "faithfulness": int(f_match.group(1)) if f_match else 3,
            "relevance": int(r_match.group(1)) if r_match else 3,
            "completeness": int(c_match.group(1)) if c_match else 3,
            "verdict": v_match.group(1) if v_match else "fail",
            "notes": "Score extracted with fallback parsing",
        }


print("LLM judge functions defined.")

In [ ]:
# Evaluate a subset of 20 questions (prioritizing hard cases)
# Running all questions would take too long for an interactive notebook
hard_pairs = [p for p in all_qa_pairs if p.get("difficulty") == "hard"]
medium_pairs = [p for p in all_qa_pairs if p.get("difficulty") == "medium"]
easy_pairs = [p for p in all_qa_pairs if p.get("difficulty") == "easy"]

eval_subset = (hard_pairs[:6] + medium_pairs[:8] + easy_pairs[:6])[:20]

print("=" * 65)
print("  Step 3: LLM Judge — Answer Quality Evaluation")
print("=" * 65)
print(f"\nEvaluating {len(eval_subset)} questions "
      f"({len([p for p in eval_subset if p.get('difficulty') == 'hard'])} hard, "
      f"{len([p for p in eval_subset if p.get('difficulty') == 'medium'])} medium, "
      f"{len([p for p in eval_subset if p.get('difficulty') == 'easy'])} easy)")

judgments = []

for i, pair in enumerate(eval_subset, 1):
    question = pair["question"]
    expected = pair.get("expected_answer", "")

    # Generate answer
    answer, context = generate_answer(question)

    # Judge answer
    scores = judge_answer(question, expected, context, answer)

    judgments.append({
        "question": question,
        "difficulty": pair.get("difficulty", "unknown"),
        "source_file": pair["source_file"],
        "answer": answer,
        "scores": scores,
    })

    verdict_icon = "OK" if scores["verdict"] == "pass" else "FAIL"
    print(f"  [{i}/{len(eval_subset)}] {verdict_icon} F:{scores['faithfulness']} "
          f"R:{scores['relevance']} C:{scores['completeness']} "
          f"| [{pair.get('difficulty','?')}] {question[:50]}...")

In [ ]:
# Results summary
pass_count = sum(1 for j in judgments if j["scores"]["verdict"] == "pass")
fail_count = len(judgments) - pass_count

avg_faith = sum(j["scores"]["faithfulness"] for j in judgments) / len(judgments)
avg_relev = sum(j["scores"]["relevance"] for j in judgments) / len(judgments)
avg_compl = sum(j["scores"]["completeness"] for j in judgments) / len(judgments)

print("\n" + "=" * 65)
print("  LLM Judge Results Summary")
print("=" * 65)

print(f"\nOverall results ({len(judgments)} questions):")
print(f"  Pass: {pass_count} ({pass_count/len(judgments)*100:.0f}%)")
print(f"  Fail: {fail_count} ({fail_count/len(judgments)*100:.0f}%)")

print(f"\nAverage scores (out of 5):")
print(f"  Faithfulness:  {avg_faith:.2f}/5  (Are answers grounded in context?)")
print(f"  Relevance:     {avg_relev:.2f}/5  (Do answers address the question?)")
print(f"  Completeness:  {avg_compl:.2f}/5  (Do answers cover all aspects?)")

# Per-difficulty breakdown
print("\nBy difficulty:")
for diff in ["easy", "medium", "hard"]:
    diff_results = [j for j in judgments if j.get("difficulty") == diff]
    if diff_results:
        diff_pass = sum(1 for j in diff_results if j["scores"]["verdict"] == "pass")
        print(f"  {diff.capitalize()}: {diff_pass}/{len(diff_results)} pass "
              f"({diff_pass/len(diff_results)*100:.0f}%)")

# Show failures
failures = [j for j in judgments if j["scores"]["verdict"] == "fail"]
if failures:
    print(f"\nFailed Answers ({len(failures)}):")
    for fail in failures[:3]:
        print(f"\n  Q: {fail['question'][:65]}")
        print(f"  Scores: F:{fail['scores']['faithfulness']} "
              f"R:{fail['scores']['relevance']} C:{fail['scores']['completeness']}")
        print(f"  Notes: {fail['scores']['notes']}")

## Step 4: RAGAS Evaluation

**RAGAS** (Retrieval Augmented Generation Assessment) is an open-source framework that computes four metrics:

| Metric | Question it answers |
|---|---|
| Context Precision | Of the retrieved chunks, what fraction were actually useful? |
| Context Recall | Of all information needed, what fraction did we retrieve? |
| Faithfulness | Does the answer contradict the retrieved context? |
| Answer Relevancy | Is the answer relevant to the question asked? |

Together these give a **comprehensive view** of both retrieval quality (precision + recall) and generation quality (faithfulness + relevancy).

In [ ]:
# Check if RAGAS is available
try:
    from ragas import evaluate
    from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
    from datasets import Dataset
    RAGAS_AVAILABLE = True
    print("RAGAS is available.")
except ImportError:
    RAGAS_AVAILABLE = False
    print("RAGAS not installed. Install with: pip install ragas langchain-anthropic")
    print("The cell below will show expected results and metric explanations.")

try:
    from langchain_anthropic import ChatAnthropic
    LANGCHAIN_AVAILABLE = True
except (ImportError, AttributeError):
    LANGCHAIN_AVAILABLE = False

In [ ]:
from utils import hybrid_search, ask_claude

RAGAS_ANSWER_SYSTEM = """You are a helpful NovaDash support assistant.
Answer the question using ONLY the documentation provided. Be specific and direct."""


def build_ragas_dataset(eval_pairs: list, n: int = 15) -> list:
    """
    Build the RAGAS dataset format from our eval pairs.
    RAGAS expects: question, answer, contexts (list of strings), ground_truth
    """
    subset = eval_pairs[:n]
    ragas_rows = []

    print(f"Building RAGAS dataset from {len(subset)} questions...")

    for i, pair in enumerate(subset, 1):
        question = pair["question"]
        print(f"  [{i}/{len(subset)}] {question[:55]}...", end="", flush=True)

        hits = hybrid_search(question, top_k=4)
        contexts = [h["text"] for h in hits]  # RAGAS expects a list of strings

        context_str = "\n\n---\n\n".join([
            f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
            for h in hits
        ])
        answer = ask_claude(
            system=RAGAS_ANSWER_SYSTEM,
            user=f"Documentation context:\n{context_str}\n\nQuestion: {question}",
            model="claude-haiku-4-5-20251001",
        )

        ragas_rows.append({
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": pair.get("expected_answer", ""),
        })
        print(" done")

    return ragas_rows


# Build the dataset
ragas_rows = build_ragas_dataset(all_qa_pairs, n=15)

In [ ]:
if RAGAS_AVAILABLE and LANGCHAIN_AVAILABLE:
    # Run RAGAS evaluation (uses Claude as the judge LLM)
    import keyring
    api_key = keyring.get_password("Anthorpic", "API_KEY")

    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=api_key)
    dataset = Dataset.from_list(ragas_rows)

    print("Running RAGAS evaluation (many API calls — may take 2-5 minutes)...\n")
    result = evaluate(
        dataset=dataset,
        metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
        llm=llm,
        raise_exceptions=False,
    )

    df = result.to_pandas()
    print("\n" + "=" * 65)
    print("  RAGAS Evaluation Report")
    print("=" * 65)
    print(f"\nOverall Scores:")
    print(f"  Context Precision:   {df['context_precision'].mean():.3f}")
    print(f"  Context Recall:      {df['context_recall'].mean():.3f}")
    print(f"  Faithfulness:        {df['faithfulness'].mean():.3f}")
    print(f"  Answer Relevancy:    {df['answer_relevancy'].mean():.3f}")

else:
    # RAGAS not installed — show what we built and expected results
    print("\n" + "=" * 65)
    print("  RAGAS Not Installed — Dataset Summary")
    print("=" * 65)
    print(f"\nBuilt RAGAS dataset with {len(ragas_rows)} rows.")
    print("\nTo run RAGAS: pip install ragas langchain-anthropic")
    print()
    print("RAGAS metrics that would be computed:")
    print("  context_precision  — fraction of retrieved chunks actually useful")
    print("  context_recall     — fraction of needed information that was retrieved")
    print("  faithfulness       — fraction of claims supported by context")
    print("  answer_relevancy   — how well the answer addresses the question")
    print()
    print("Expected results based on Chapter 6 hybrid search (approximate):")
    print("  Context Precision:  0.65 - 0.75")
    print("  Context Recall:     0.70 - 0.80")
    print("  Faithfulness:       0.85 - 0.95")
    print("  Answer Relevancy:   0.80 - 0.90")

    print("\nFirst 3 questions in the RAGAS dataset:")
    for row in ragas_rows[:3]:
        print(f"\n  Q: {row['question'][:65]}")
        print(f"  A: {row['answer'][:120].strip()}...")
        print(f"  Contexts: {len(row['contexts'])} chunks retrieved")

## How to Interpret RAGAS Metrics

| Metric | Score | Diagnosis | Fix |
|---|---|---|---|
| Context Precision | Low (<0.6) | Retrieving too many irrelevant chunks | Reduce top_k or add reranking |
| Context Recall | Low (<0.6) | Missing relevant chunks | Increase top_k or use hybrid/multi-query |
| Faithfulness | Low (<0.7) | Possible hallucination | Stronger grounding prompts (Chapter 5) |
| Answer Relevancy | Low (<0.7) | Answers are off-topic | Improve retrieval so context is more focused |

The power of RAGAS is that it tells you **which part** of your pipeline is broken:
- Low precision + high recall → retrieval noise (fix: reranker)
- Low recall + high precision → retrieval gap (fix: hybrid/multi-query)
- Low faithfulness → hallucination (fix: grounding prompts)
- Low relevancy → off-topic answers (fix: retrieval quality)

## Congratulations — You've Completed the Learning Track!

```
  You now know how to:

  Chapter 1 — Build a basic RAG system
              Embeddings, ChromaDB, the 3-step pipeline

  Chapter 2 — Fix semantic gaps with hybrid search
              BM25 + RRF, Recall@1 from 0.25 -> 1.0 on semantic gap queries

  Chapter 3 — Improve precision with reranking
              Cross-encoder, LLM-as-reranker, contextual embeddings

  Chapter 4 — Handle complex multi-part questions
              Multi-query, decomposition, agentic ReAct loop

  Chapter 5 — Detect and prevent hallucination
              Grounding, citations, confidence scoring, guardrail stack

  Chapter 6 — Measure if it's working
              Eval datasets, Recall@K, MRR, LLM judge, RAGAS
```

These 6 chapters cover the exact problems encountered building real production AI systems for healthcare and SaaS platforms.

**The highest-leverage insight from all of this:**
> Fix retrieval first. Our real-world eval showed hybrid search added +20% Recall@1. Contextual embeddings added 0%. The bottleneck was the semantic gap, not embedding quality. **Measure before optimizing.**

---

**Next:** [Production Track](../../production/) — Advanced patterns for production deployment:
- `01-langgraph` — Multi-agent orchestration
- `02-human-in-the-loop` — Confidence-gated human review
- `03-guardrails` — Production guardrail service
- `04-mcp-tools` — Tool servers for agents
- `05-eval-in-ci` — Evaluation in CI/CD pipelines